In [1]:
# 🎯 Sampling Theorem Demonstrator — Final Advanced Version
# ✅ Works directly in Google Colab
# 🧠 Features: Dual View (Time + Frequency), Aliasing Warning, Oversampling Indicator, RMS Error, Fast Reconstruction

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown, Checkbox
from IPython.display import clear_output

# -------------------------------------
# 1️⃣ Define the continuous-time signal
# -------------------------------------
def continuous_signal(t, freq, signal_type):
    if signal_type == "Sine":
        return np.sin(2 * np.pi * freq * t)
    elif signal_type == "Cosine":
        return np.cos(2 * np.pi * freq * t)
    elif signal_type == "Square":
        return np.sign(np.sin(2 * np.pi * freq * t))
    else:
        return np.sin(2 * np.pi * freq * t)

# -------------------------------------
# 2️⃣ Sampling function
# -------------------------------------
def sample_signal(t, signal_func, fs):
    Ts = 1/fs
    t_samples = np.arange(t[0], t[-1], Ts)
    y_samples = signal_func(t_samples)
    return t_samples, y_samples

# -------------------------------------
# 3️⃣ Vectorized sinc reconstruction
# -------------------------------------
def reconstruct_signal_fast(t, t_samples, y_samples, fs):
    sinc_matrix = np.sinc((t[:, None] - t_samples[None, :]) * fs)
    return np.dot(sinc_matrix, y_samples)

# -------------------------------------
# 4️⃣ Frequency spectrum (FFT)
# -------------------------------------
def compute_spectrum(signal, fs):
    N = len(signal)
    freq = np.fft.fftfreq(N, 1/fs)
    magnitude = np.abs(np.fft.fft(signal)) / N
    return freq[:N//2], magnitude[:N//2]

# -------------------------------------
# 5️⃣ Main plotting function (dual view)
# -------------------------------------
def plot_sampling(signal_freq=5.0, sampling_freq=10.0, signal_type="Sine",
                  show_reconstruction=True, dual_view=True):

    clear_output(wait=True)
    t = np.linspace(0, 1, 1000)
    y_cont = continuous_signal(t, signal_freq, signal_type)
    t_samples, y_samples = sample_signal(t, lambda tt: continuous_signal(tt, signal_freq, signal_type), sampling_freq)
    y_recon = reconstruct_signal_fast(t, t_samples, y_samples, sampling_freq) if show_reconstruction else np.zeros_like(y_cont)

    # --- Time Domain Plot ---
    if dual_view:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        ax1, ax2 = axes
    else:
        fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.plot(t, y_cont, label="Continuous Signal", color='blue')
    ax1.plot(t_samples, y_samples, 'o', label="Sampled Points", color='red')
    if show_reconstruction:
        ax1.plot(t, y_recon, '--', label="Reconstructed Signal", color='green')
    ax1.set_title("Time-Domain Representation")
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Amplitude")
    ax1.grid(True)
    ax1.legend()

    # Aliasing / Proper Sampling / Oversampling Indicator
    if sampling_freq < 2 * signal_freq:
        ax1.text(0.02, 0.9, "⚠️ Under-sampling (Aliasing!)", color='red',
                 transform=ax1.transAxes, fontsize=11, fontweight='bold')

    elif sampling_freq >= 4 * signal_freq:
        ax1.text(0.02, 0.9, "🔼 Over-Sampling (Very High Fs)", color='purple',
                 transform=ax1.transAxes, fontsize=11, fontweight='bold')

    else:
        ax1.text(0.02, 0.9, "✅ Proper Sampling (No Aliasing)", color='green',
                 transform=ax1.transAxes, fontsize=11, fontweight='bold')

    ax1.text(0.02, 0.8, f"Signal Freq = {signal_freq} Hz\nSampling Freq = {sampling_freq} Hz",
             transform=ax1.transAxes, fontsize=10,
             bbox=dict(facecolor='white', alpha=0.7))

    # --- Frequency Domain Plot ---
    if dual_view:
        freq_cont, mag_cont = compute_spectrum(y_cont, 1000)
        freq_recon, mag_recon = compute_spectrum(y_recon, sampling_freq)
        ax2.plot(freq_cont, mag_cont, label="Original Spectrum", color='blue')
        if show_reconstruction:
            ax2.plot(freq_recon, mag_recon, '--',
                     label="Reconstructed Spectrum", color='green')
        ax2.set_title("Frequency-Domain Representation")
        ax2.set_xlabel("Frequency (Hz)")
        ax2.set_ylabel("Magnitude")
        ax2.grid(True)
        ax2.legend()

    plt.tight_layout()
    plt.show()

    # RMS Error (if reconstruction is enabled)
    if show_reconstruction:
        error = np.sqrt(np.mean((y_cont - y_recon)**2))
        print(f"🔹 RMS Reconstruction Error: {error:.4f}")

# -------------------------------------
# 6️⃣ Interactive Controls
# -------------------------------------
interact(
    plot_sampling,
    signal_freq=FloatSlider(value=5, min=1, max=20, step=0.5, description="Signal Freq (Hz)"),
    sampling_freq=FloatSlider(value=10, min=1, max=75, step=1, description="Sampling Freq (Hz)"),
    signal_type=Dropdown(options=["Sine", "Cosine", "Square"], value="Sine", description="Signal Type"),
    show_reconstruction=Checkbox(value=True, description="Show Reconstruction"),
    dual_view=Checkbox(value=True, description="Dual View (Time + Freq)")
)


interactive(children=(FloatSlider(value=5.0, description='Signal Freq (Hz)', max=20.0, min=1.0, step=0.5), Flo…

<function __main__.plot_sampling(signal_freq=5.0, sampling_freq=10.0, signal_type='Sine', show_reconstruction=True, dual_view=True)>